# Aufteilung der Medienbeiträge

Die JSON-Datei liegt im **übergeordneten Ordner** (unabhängig vom Ordnernamen). Schritte:

1. **Sprache:** Deutsch und Französisch
2. **Kanal:** Zeitungen, Social Media (nur relevante Plattformen), **Urlaub**, **Regionalnachrichten**, sonst Andere Webquellen
3. **Social-Media-Quelle:** YouTube, Instagram, Facebook … (ohne Tumblr, Telegram, Pinterest, Bluesky, Vimeo)

### Kanal-Regeln

- **Social Media:** nur Plattformen mit ausreichend Relevanz (ab Niveau TikTok/LinkedIn/Reddit nach oben: YouTube, Instagram, Facebook, Mastodon, Flickr, Reddit, LinkedIn, TikTok, X)
- **Urlaub:** Reise-/Ski-/Wanderportale, Foren, typische Urlaubsbegriffe oder passende Bildquellen
- **Regionalnachrichten:** regionale Portale und Kommentare (ohne `News`, die bereits unter Zeitungen liegen)
- **Zeitungen:** `media_type_name == News`
- **Andere Webquellen:** Rest


In [ ]:
from pathlib import Path
import re

import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.ticker import FuncFormatter

DATEN_NAME = "FSI_Heilbronn_Medienbeiträge (1).json"

SOCIAL_RELEVANT = frozenset({
    "YouTube", "Instagram", "Facebook", "Mastodon", "Flickr", "Reddit", "LinkedIn", "TikTok", "X",
})

PLATTFORM_DOMAINS = (
    ("YouTube", ("youtube.com", "youtu.be")),
    ("Instagram", ("instagram.com",)),
    ("Facebook", ("facebook.com",)),
    ("Reddit", ("reddit.com",)),
    ("LinkedIn", ("linkedin.com",)),
    ("TikTok", ("tiktok.com",)),
    ("Telegram", ("t.me", "telegram.me", "telegram.org")),
    ("Flickr", ("flickr.com",)),
    ("Pinterest", ("pinterest.com",)),
    ("Tumblr", ("tumblr.com",)),
    ("Bluesky", ("bsky.app",)),
    ("Vimeo", ("vimeo.com",)),
    ("X", ("twitter.com", "x.com")),
)

URLAUB_HOST_MARKEN = (
    "erlebnisrundreisen", "holidaycheck", "urlaubspiraten", "hometravel", "phototravel",
    "alpinforum", "skipass", "ski-nordique", "bergrettung", "22places", "kommwirmachendas",
    "urlaub", "partir.", "tui.com", "mein-mmo", "flugreisende",
)

REGIONAL_HOST_MARKEN = (
    "meinbezirk", "bezirk", "anzeiger", "rundblick", "24hamburg", "bw24", "soester-anzeiger",
    "sauerlandkurier", "salzkammergut", "rheingau", "az-online", "wmn.de", "it-boltwise",
    "vol.at", "krone.at", "kurier.at",
)

KANAL_REIHENFOLGE = (
    "Zeitungen und klassische Medien",
    "Social Media",
    "Urlaub",
    "Regionalnachrichten",
    "Andere Webquellen",
)
SPRACHEN = ("Deutsch", "Französisch")
FARBEN = {"Deutsch": "#e07a3d", "Französisch": "#3d7ec9"}


def datenordner():
    hier = Path.cwd().resolve()
    for ordner in (hier.parent, hier, hier.parent.parent):
        if (ordner / DATEN_NAME).is_file():
            return ordner
    return hier.parent


DATEN = datenordner() / DATEN_NAME


def host_aus_quelle(quelle):
    host = str(quelle or "").strip().lower()
    if host.startswith("www."):
        host = host[4:]
    return host


def host_ist(host, domain):
    return host == domain or host.endswith("." + domain)


def plattform_roh(quelle, medientyp):
    host = host_aus_quelle(quelle)
    for name, domains in PLATTFORM_DOMAINS:
        if any(host_ist(host, domain) for domain in domains):
            return name
    if medientyp == "Microblog":
        return "Mastodon"
    if medientyp in {"YouTube Video", "YouTube Comments"}:
        return "YouTube"
    return None


def plattform_bestimmen(quelle, medientyp):
    name = plattform_roh(quelle, medientyp)
    if name in SOCIAL_RELEVANT:
        return name
    return None


def sprache_bestimmen(code):
    code = str(code or "").lower()
    if code.startswith("de"):
        return "Deutsch"
    if code.startswith("fr"):
        return "Französisch"
    return "Andere"


def urlaub_bestimmen(host, medientyp, titel, inhalt):
    text = f"{titel}\n{inhalt}".lower()[:2500]
    if any(marke in host for marke in URLAUB_HOST_MARKEN):
        return True
    if medientyp == "Forums":
        return True
    if medientyp == "Pictures" and "infoclimat" not in host:
        if any(w in text for w in ("wander", "ski", "urlaub", "reise", "bergbahn", "gondel")):
            return True
    treffer = sum(1 for w in ("urlaub", "reise", "wanderung", "skigebiet", "ferien", "bergfahrt", "hotel") if w in text)
    if treffer >= 2:
        return True
    return "urlaub" in text and medientyp in {"Blog", "Other"}


def regional_bestimmen(host, medientyp):
    if medientyp == "News":
        return False
    if any(marke in host for marke in REGIONAL_HOST_MARKEN):
        return True
    return medientyp in {"Other", "Comments", "Blog"} and any(
        marke in host for marke in ("krone.at", "vol.at", "kurier.at", "meinbezirk")
    )


def kanal_bestimmen(medientyp, plattform, host, titel, inhalt):
    if isinstance(plattform, str) and plattform:
        return "Social Media"
    if medientyp == "News":
        return "Zeitungen und klassische Medien"
    if urlaub_bestimmen(host, medientyp, titel or "", inhalt or ""):
        return "Urlaub"
    if regional_bestimmen(host, medientyp):
        return "Regionalnachrichten"
    return "Andere Webquellen"


def mit_summe(tabelle):
    ergebnis = tabelle.copy()
    ergebnis["Summe"] = ergebnis.sum(axis=1)
    ergebnis.loc["Summe"] = ergebnis.sum(axis=0)
    return ergebnis.astype(int)


def als_text(tabelle):
    def zelle(wert):
        return f"{int(wert):,}".replace(",", ".")

    if isinstance(tabelle, pd.Series):
        return tabelle.map(zelle)
    return tabelle.apply(lambda spalte: spalte.map(zelle))


def achsen_tausender(achse, _position):
    return f"{int(achse):,}".replace(",", ".")


df = pd.read_json(DATEN, orient="records")
rows = []
for zeile in df.itertuples(index=False):
    quelle = zeile.source_name
    medientyp = zeile.media_type_name
    host = host_aus_quelle(quelle)
    plattform = plattform_bestimmen(quelle, medientyp)
    plattform_hinweis = plattform_roh(quelle, medientyp)
    rows.append(
        {
            "sprache": sprache_bestimmen(zeile.language_code),
            "plattform": plattform,
            "plattform_hinweis": plattform_hinweis,
            "kanal": kanal_bestimmen(medientyp, plattform, host, zeile.title, zeile.content),
        }
    )

zusatz = pd.DataFrame(rows, index=df.index)
df = pd.concat([zusatz, df.drop(columns=[c for c in zusatz.columns if c in df.columns], errors="ignore")], axis=1)

vorn = [
    "sprache", "kanal", "plattform", "plattform_hinweis",
    "media_type_name", "source_name", "language_code", "published_at", "title",
]
df = df[vorn + [spalte for spalte in df.columns if spalte not in vorn]]

nicht_relevant_social = df["plattform"].isna() & df["plattform_hinweis"].notna()
print("Social ohne Relevanz (z. B. Tumblr, Telegram):", int(nicht_relevant_social.sum()))

df[vorn].head(8)



## Sprache


In [ ]:
sprache_anzahl = df["sprache"].value_counts().rename("Beiträge")
sprache_anzahl = sprache_anzahl.reindex(["Deutsch", "Französisch", "Andere"])
als_text(sprache_anzahl.to_frame())



In [ ]:
fig, ax = plt.subplots(figsize=(7.2, 4.2))
balken = sprache_anzahl.loc[list(SPRACHEN)]
balken.plot(kind="bar", ax=ax, color=[FARBEN[n] for n in balken.index], width=0.7)
ax.set_title("Beiträge nach Sprache")
ax.set_ylabel("Beiträge")
ax.yaxis.set_major_formatter(FuncFormatter(achsen_tausender))
fig.tight_layout()
plt.show()



## Kanäle inkl. Urlaub und Regionalnachrichten


In [ ]:
teil = df[df["sprache"].isin(SPRACHEN)]
kanal_anzahl = (
    pd.crosstab(teil["kanal"], teil["sprache"])
    .reindex(index=list(KANAL_REIHENFOLGE), columns=list(SPRACHEN), fill_value=0)
)
als_text(mit_summe(kanal_anzahl))



In [ ]:
fig, ax = plt.subplots(figsize=(9.5, 4.8))
kanal_anzahl.plot(kind="bar", ax=ax, color=[FARBEN[n] for n in kanal_anzahl.columns], width=0.75)
ax.set_title("Kanal nach Sprache")
ax.set_ylabel("Beiträge")
ax.tick_params(axis="x", rotation=18)
ax.yaxis.set_major_formatter(FuncFormatter(achsen_tausender))
fig.tight_layout()
plt.show()



In [ ]:
def ausschnitt(sprache, kanal=None, plattform=None):
    auswahl = df[df["sprache"].eq(sprache)]
    if kanal is not None:
        auswahl = auswahl[auswahl["kanal"].eq(kanal)]
    if plattform is not None:
        auswahl = auswahl[auswahl["plattform"].eq(plattform)]
    return auswahl

for name in ("Urlaub", "Regionalnachrichten", "Andere Webquellen"):
    teilmenge = df[df["sprache"].isin(SPRACHEN) & df["kanal"].eq(name)]
    tabelle = (
        pd.crosstab(teilmenge["media_type_name"], teilmenge["sprache"])
        .reindex(columns=list(SPRACHEN), fill_value=0)
        .sort_values("Deutsch", ascending=False)
    )
    print(f"\n=== {name} ===")
    print(als_text(mit_summe(tabelle)).to_string())



## Social Media (nur relevante Plattformen)


In [ ]:
sozial = df[df["kanal"].eq("Social Media") & df["sprache"].isin(SPRACHEN)]
plattform_anzahl = pd.crosstab(sozial["plattform"], sozial["sprache"]).reindex(columns=list(SPRACHEN), fill_value=0)
plattform_anzahl["Summe"] = plattform_anzahl.sum(axis=1)
plattform_anzahl = plattform_anzahl.sort_values("Summe", ascending=False)
als_text(mit_summe(plattform_anzahl.astype(int)))



In [ ]:
plot_plattform = plattform_anzahl.drop(columns="Summe").sort_values("Deutsch", ascending=True)
fig, ax = plt.subplots(figsize=(8.4, 5.0))
plot_plattform.plot(kind="barh", ax=ax, color=[FARBEN[n] for n in plot_plattform.columns], width=0.75)
ax.set_title("Relevante Social-Media-Quellen")
ax.set_xlabel("Beiträge")
ax.xaxis.set_major_formatter(FuncFormatter(achsen_tausender))
fig.tight_layout()
plt.show()



In [ ]:
ausgeschlossen = (
    df[df["plattform"].isna() & df["plattform_hinweis"].notna()]
    .groupby("plattform_hinweis")
    .size()
    .sort_values(ascending=False)
    .rename("Beiträge")
)
print("Nicht als Social Media gezählt (unter Relevanzschwelle):")
ausgeschlossen



In [ ]:
youtube = df[df["plattform"].eq("YouTube") & df["sprache"].isin(SPRACHEN)].copy()
youtube["beitragsart"] = youtube["media_type_name"].map(
    {"YouTube Video": "Video", "YouTube Comments": "Kommentar"}
).fillna("Sonstige")
youtube_anzahl = pd.crosstab(youtube["beitragsart"], youtube["sprache"]).reindex(
    index=["Video", "Kommentar", "Sonstige"], columns=list(SPRACHEN), fill_value=0
)
youtube_anzahl = youtube_anzahl.loc[youtube_anzahl.sum(axis=1).gt(0)]
als_text(mit_summe(youtube_anzahl))

